# STAGE 2: fine-tune L2/L3/L4 + PairHead from the L3/L4 control (T=10, film3/head3)

Loads the stage-1 L3/L4 control checkpoint (PairHead trained with L1-L4 frozen) and fine-tunes with L1 frozen but L2 (cross) + L3 (dual-role) + L4 (joint-role) + PairHead all trainable at a 5× lower LR (2e-5), so the upper stack co-adapts to the trained head. L0 stays frozen. Everything else matches stage 1 (T=10, film3/head3, same pair_cache_t10).

Stack being trained:

```
L0 (planet/fleet/comet)   frozen, no_grad
      ↓
L1 PlanetEntityEncoder    warm-started (stage-1 ckpt), frozen via --freeze-l1-only
      ↓
L2 CrossEntityAttention   warm-started, TRAINS
      ↓ ctx_now (B, P, 256)
      ↓ L3 DualRoleAttention  (warm-started, TRAINS)
      ↓ L4 JointRoleAttention (warm-started, TRAINS)
PlayerConsolidator        omitted via --no-consolidator (unused by pair supervision)
PairHead                  TRAINS
   trunk (3-Linear MLP) + FiLM (conditioner depth=3) + 2 heads (depth=3)
      ↓
pair_logits (B, P, P)     BCE, pos_weight=600
pair_frac   (B, P, P)     MSE on positive cells
```

Warm-start: the stage-1 control run `L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_20260529-084115` (its `entity_encoder_best.pt` on GCS), pulled into `INIT_FROM_ENTITY`.

## Inputs from GCS

```
gs://orbit-wars-shipping/entity/
  code.tgz                  # agents/ + scripts/ (same bundle as stage 1)
  weights.tgz               # frozen L0 (planet, fleet, comet) d=256
  pair_cache_t10.part_*     # preferred: chunked patched cache with T=10 offsets
  pair_cache_t10.manifest.json
  pair_cache_t10.pt         # fallback: single-object patched cache
  runs/L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_.../entity_encoder_best.pt
```


## 1. Authenticate + pull bundle

In [ ]:
from google.colab import auth
auth.authenticate_user()
BUCKET = 'gs://orbit-wars-shipping/entity'
print(f'pulling from {BUCKET}')

In [ ]:
import os, subprocess, time, json, hashlib, concurrent.futures
from pathlib import Path

WORK = Path('/content/orbit-wars')
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

PAIR_CACHE = WORK / 'pair_cache.pt'
PAIR_CACHE_PREFIX = 'pair_cache_t10'
MANIFEST_LOCAL = WORK / f'{PAIR_CACHE_PREFIX}.manifest.json'

def _gcs_size(url: str) -> int | None:
    try:
        out = subprocess.run(
            ['gcloud', 'storage', 'objects', 'describe', url, '--format=value(size)'],
            check=True, capture_output=True, text=True,
        )
    except subprocess.CalledProcessError:
        return None
    try:
        return int(out.stdout.strip())
    except (TypeError, ValueError):
        return None

def cp(src, dst, *, force=True):
    dst = Path(dst)
    if dst.exists() and not force:
        return dst.name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    t0 = time.time()
    print(f'  pulling {src} → {dst.name} ...', flush=True)
    subprocess.run(['gcloud', 'storage', 'cp', src, str(dst)], check=True)
    return dst.name, time.time() - t0, dst.stat().st_size

def _sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for blk in iter(lambda: fh.read(1 << 20), b''):
            h.update(blk)
    return h.hexdigest()

def _fetch_manifest() -> dict | None:
    remote = f'{BUCKET}/{PAIR_CACHE_PREFIX}.manifest.json'
    if _gcs_size(remote) is None:
        return None
    cp(remote, MANIFEST_LOCAL)
    return json.loads(MANIFEST_LOCAL.read_text())

def _pull_chunk(spec: dict, chunks_dir: Path):
    name = spec['name']
    dst = chunks_dir / name
    expected_size = int(spec.get('size_bytes', 0))
    expected_sha = spec.get('sha256')
    if dst.exists() and (not expected_size or dst.stat().st_size == expected_size):
        if expected_sha is None or _sha256_file(dst) == expected_sha:
            return name, 0.0, dst.stat().st_size
    if dst.exists():
        dst.unlink()
    name, dt, size = cp(f'{BUCKET}/{name}', dst)
    if expected_sha and _sha256_file(dst) != expected_sha:
        raise RuntimeError(f'sha256 mismatch on {name}')
    return name, dt, size

def _assemble_chunks(manifest: dict, chunks_dir: Path):
    chunks = [c['name'] for c in manifest['chunks']]
    total_bytes = int(manifest.get('total_bytes', 0))
    if PAIR_CACHE.exists():
        PAIR_CACHE.unlink()
    print(f'  assembling {len(chunks)} chunks → pair_cache.pt ...', flush=True)
    t0 = time.time()
    with open(PAIR_CACHE, 'wb') as out_fh:
        for name in chunks:
            with open(chunks_dir / name, 'rb') as in_fh:
                for blk in iter(lambda: in_fh.read(1 << 22), b''):
                    out_fh.write(blk)
    actual = PAIR_CACHE.stat().st_size
    if total_bytes and actual != total_bytes:
        raise RuntimeError(f'assembled size mismatch: got {actual}, expected {total_bytes}')
    return time.time() - t0, actual

def _pull_pair_cache():
    t0 = time.time()
    manifest = _fetch_manifest()
    if manifest is None:
        remote = f'{BUCKET}/{PAIR_CACHE_PREFIX}.pt'
        remote_size = _gcs_size(remote)
        if remote_size is None:
            raise RuntimeError(f'no {PAIR_CACHE_PREFIX}.manifest.json or {PAIR_CACHE_PREFIX}.pt on {BUCKET}')
        if PAIR_CACHE.exists() and PAIR_CACHE.stat().st_size == remote_size:
            return f'{PAIR_CACHE_PREFIX}.pt (single cached)', 0.0, remote_size
        return cp(remote, PAIR_CACHE)

    total = int(manifest.get('total_bytes', 0))
    if PAIR_CACHE.exists() and total and PAIR_CACHE.stat().st_size == total:
        return f'{PAIR_CACHE_PREFIX} chunks (assembled cached)', 0.0, total
    chunks_dir = WORK / f'{PAIR_CACHE_PREFIX}_chunks'
    chunks_dir.mkdir(parents=True, exist_ok=True)
    specs = manifest['chunks']
    print(f'  chunked pair cache: {len(specs)} chunks, {total/1024**3:.2f} GB total')
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(specs)) as pool:
        futs = [pool.submit(_pull_chunk, spec, chunks_dir) for spec in specs]
        for fut in concurrent.futures.as_completed(futs):
            cname, cdt, csize = fut.result()
            print(f'    {cname:<28s} {csize/1024**2:>8.1f} MB  {cdt:5.1f}s')
    assemble_dt, actual = _assemble_chunks(manifest, chunks_dir)
    return f'{PAIR_CACHE_PREFIX} chunks', time.time() - t0, actual

TASKS = [
    (f'{BUCKET}/code.tgz',    WORK / 'code.tgz'),
    (f'{BUCKET}/weights.tgz', WORK / 'weights.tgz'),
]
T_START = time.time()
results = []
with concurrent.futures.ThreadPoolExecutor(max_workers=len(TASKS) + 1) as pool:
    futs = [pool.submit(cp, s, d) for s, d in TASKS]
    futs.append(pool.submit(_pull_pair_cache))
    for f in concurrent.futures.as_completed(futs):
        results.append(f.result())
for name, dt, size in sorted(results, key=lambda t: -t[2]):
    mb = size / 1024 / 1024
    print(f'  {name:<26s} {mb:>9.1f} MB  in {dt:5.1f}s')
print(f'\ntotal wall: {time.time()-T_START:.1f}s')

In [ ]:
# Wipe stale extracted code so a previous bundle can't shadow the new one.
!rm -rf agents scripts ckpts
!find . -maxdepth 1 -name '*.pt' ! -name 'pair_cache.pt' -delete

!tar xzf code.tgz
!tar xzf weights.tgz

import sys
for m in list(sys.modules):
    if m.startswith('agents') or m.startswith('scripts'):
        del sys.modules[m]
import importlib, gc
importlib.invalidate_caches()
gc.collect()
!find . -type d -name __pycache__ -exec rm -rf {} + 2>/dev/null || true
!ls -la

## 1b. Verify L3/L4 ACTIVE + T=10 cache config

In [ ]:
from agents.transformer_v2.pretrain.entity_encoder import EntityPretrainModel
m = EntityPretrainModel(d_model=256, n_steps=10, conditioner_n_layers=3, head_n_layers=3, skip_l34=False)
assert m.dual_role is not None and m.joint_role is not None, 'L3/L4 should be present in the control'
print(f'L3/L4 active: dual_role={type(m.dual_role).__name__}, joint_role={type(m.joint_role).__name__}')


## 2. Stage the cache where the CLI expects it

In [ ]:
from pathlib import Path
import os
CACHE_DIR = Path('data/datasets/_pair_cache/bowwowforeach_Ebi_T6')
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_TARGET = CACHE_DIR / 'bowwowforeach_Ebi_T6_p64_f1024_all.pt'
if CACHE_TARGET.exists():
    CACHE_TARGET.unlink()
os.link('/content/orbit-wars/pair_cache.pt', CACHE_TARGET)
PAIR_CACHE_PATH = str(CACHE_TARGET)
print(f'staged: {PAIR_CACHE_PATH}  ({CACHE_TARGET.stat().st_size/1024**3:.2f} GB)')

## 3. Verify GPU

In [ ]:
import torch
print(f'torch: {torch.__version__}, cuda available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device: {torch.cuda.get_device_name(0)}')
    print(f'  mem total: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

## 4. Stage L0 ckpts + warm-start entity ckpt

In [ ]:
import shutil, subprocess
from pathlib import Path
PLANET_RUN_DIR = Path('/content/orbit-wars/ckpts/planet')
FLEET_RUN_DIR  = Path('/content/orbit-wars/ckpts/fleet')
COMET_RUN_DIR  = Path('/content/orbit-wars/ckpts/comet')
for d in (PLANET_RUN_DIR, FLEET_RUN_DIR, COMET_RUN_DIR):
    d.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/orbit-wars/planet_encoder_best.pt', PLANET_RUN_DIR / 'planet_encoder_best.pt')
shutil.copy('/content/orbit-wars/fleet_encoder_best.pt',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt')
shutil.copy('/content/orbit-wars/comet_past_best.pt',     COMET_RUN_DIR  / 'comet_past_best.pt')

BASELINE_RUN = 'L3L4_T10_film3_head3_d256_b256_20ep_lr0.0001_20260529-084115'
BASELINE_DIR = Path(f'/content/orbit-wars/ckpts/baseline/{BASELINE_RUN}')
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_CKPT = BASELINE_DIR / 'entity_encoder_best.pt'
if not BASELINE_CKPT.exists():
    subprocess.run(
        ['gcloud', 'storage', 'cp',
         f'{BUCKET}/runs/{BASELINE_RUN}/entity_encoder_best.pt',
         str(BASELINE_CKPT)],
        check=True,
    )
print(f'baseline ckpt: {BASELINE_CKPT}  ({BASELINE_CKPT.stat().st_size/1024**2:.1f} MB)')

import torch
for tag, p in (('planet', PLANET_RUN_DIR / 'planet_encoder_best.pt'),
                ('fleet',  FLEET_RUN_DIR  / 'fleet_encoder_best.pt'),
                ('comet',  COMET_RUN_DIR  / 'comet_past_best.pt')):
    c = torch.load(p, map_location='cpu', weights_only=False)
    print(f'{tag:6s} ckpt: d_model={c["config"]["d_model"]}, epoch={c["epoch"]}')
    assert c['config']['d_model'] == 256

Full stack: L3 `DualRoleAttention` + L4 `JointRoleAttention` are **built and warm-started** from the May-21 ckpt. `--no-consolidator` omits the unused PlayerConsolidator. `--freeze-perception` freezes L1+L2+L3+L4 (all warm-started); only PairHead trains. The sole difference from the no-L3/L4 ablation is that PairHead reads L4's `source_joint`/`target_joint` instead of L2's `ctx_now` twice.


In [ ]:
D_MODEL              = 256
D_PAIR               = 256
ENTITY_N_HEADS       = 8
CROSS_N_HEADS        = 8
CROSS_N_LAYERS       = 2
DUAL_N_HEADS         = 8   # L3 DualRole + L4 JointRole attention heads (ACTIVE in this control)
CONDITIONER_N_LAYERS = 3
HEAD_N_LAYERS        = 3
BATCH_SIZE           = 256   # match the stage-1 L3/L4 control run (b256)
NUM_WORKERS          = 2    # DataLoader workers for cached T-stacking
EPOCHS               = 20
LR                   = 2e-5
WEIGHT_DECAY         = 1e-4
SEED                 = 1729
MAX_PLANETS          = 64
MAX_FLEETS           = 1024
PAIR_POS_WEIGHT      = 600.0
VAL_FRAC             = 0.10
TEST_FRAC            = 0.10
DEVICE               = 'cuda' if __import__('torch').cuda.is_available() else 'cpu'
INIT_FROM_ENTITY     = str(BASELINE_CKPT)

import time
TS = time.strftime('%Y%m%d-%H%M%S')
RUN_TAG = f'L3L4_ft_l234_T10_film{CONDITIONER_N_LAYERS}_head{HEAD_N_LAYERS}_d{D_MODEL}_b{BATCH_SIZE}_{EPOCHS}ep_lr{LR:g}_{TS}'
OUT_DIR = f'data/runs/entity/{RUN_TAG}'
print('out dir:', OUT_DIR)
print(f'warm-start: {INIT_FROM_ENTITY}')

In [ ]:
!python -u -m agents.transformer_v2.pretrain.entity_encoder \
  --planet-run-dir $PLANET_RUN_DIR \
  --fleet-run-dir  $FLEET_RUN_DIR \
  --comet-run-dir  $COMET_RUN_DIR \
  --pair-cache-path $PAIR_CACHE_PATH \
  --out-dir $OUT_DIR \
  --d-model $D_MODEL \
  --d-pair $D_PAIR \
  --entity-n-heads $ENTITY_N_HEADS \
  --cross-n-heads $CROSS_N_HEADS \
  --cross-n-layers $CROSS_N_LAYERS \
  --dual-n-heads $DUAL_N_HEADS \
  --conditioner-n-layers $CONDITIONER_N_LAYERS \
  --head-n-layers $HEAD_N_LAYERS \
  --no-consolidator \
  --init-from-entity-ckpt $INIT_FROM_ENTITY \
  --freeze-l1-only \
  --batch-size $BATCH_SIZE \
  --epochs $EPOCHS \
  --lr $LR \
  --weight-decay $WEIGHT_DECAY \
  --max-planets $MAX_PLANETS \
  --max-fleets $MAX_FLEETS \
  --pair-pos-weight $PAIR_POS_WEIGHT \
  --val-frac $VAL_FRAC \
  --test-frac $TEST_FRAC \
  --num-workers $NUM_WORKERS \
  --seed $SEED \
  --device $DEVICE

## 6. Push the trained run back to GCS

In [ ]:
import subprocess
from pathlib import Path
src = Path(OUT_DIR)
assert src.is_dir(), src
dst_parent = f'{BUCKET}/runs/'
subprocess.run(['gcloud', 'storage', 'cp', '--recursive', str(src), dst_parent], check=True)
print(f'uploaded to: {dst_parent}{src.name}/')
subprocess.run(['gcloud', 'storage', 'ls', '--long', '--readable-sizes', f'{dst_parent}{src.name}/'], check=False)